In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

In [44]:
df = pd.read_csv("shop_smart_ecommerce.csv")
# NO PageValues drop — it is a valid session behaviour signal, not leakage

# Engineered features
df['avg_time_per_product'] = df['ProductRelated_Duration'] / (df['ProductRelated'] + 1)
df['total_engagement']     = (df['Administrative_Duration'] +
                               df['Informational_Duration'] +
                               df['ProductRelated_Duration'])
df['bounce_exit_score']    = df['BounceRates'] * df['ExitRates']
df['log_page_values']      = np.log1p(df['PageValues'])
df['has_page_value']       = (df['PageValues'] > 0).astype(int)
df['product_page_ratio']   = df['ProductRelated'] / (
                               df['Administrative'] + df['Informational'] +
                               df['ProductRelated'] + 1)

X = df.drop(columns=['Revenue'])
y = df['Revenue'].astype(int)

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (9864, 23) | Test: (2466, 23)


In [46]:
numeric_features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'SpecialDay', 'PageValues',
    'avg_time_per_product', 'total_engagement', 'bounce_exit_score',
    'log_page_values', 'has_page_value', 'product_page_ratio'
]
categorical_features = [
    'Month', 'OperatingSystems', 'Browser',
    'Region', 'TrafficType', 'VisitorType', 'Weekend'
]

In [47]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

In [48]:
param_grid = {
    'classifier__max_depth':        [4, 6, 8, 10],
    'classifier__min_samples_leaf': [10, 20, 30],
    'classifier__criterion':        ['gini', 'entropy'],
}

grid_search = GridSearchCV(
    pipeline, param_grid, scoring='f1', cv=5, n_jobs=-1
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print(f"Best Params : {grid_search.best_params_}")
print(f"Best CV F1  : {grid_search.best_score_:.4f}")

Best Params : {'classifier__criterion': 'entropy', 'classifier__max_depth': 4, 'classifier__min_samples_leaf': 10}
Best CV F1  : 0.6374


In [49]:
y_pred = best_model.predict(X_test)
print("--- PRODUCTION REPORT ---")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

--- PRODUCTION REPORT ---
              precision    recall  f1-score   support

           0       0.97      0.83      0.89      2084
           1       0.48      0.86      0.61       382

    accuracy                           0.83      2466
   macro avg       0.72      0.84      0.75      2466
weighted avg       0.89      0.83      0.85      2466

Confusion Matrix:
[[1727  357]
 [  55  327]]
